# VISCERA — SSL fine-tune (the method)
Domain-adapted **DINOv2 (dinov2.pth)** → fine-tune the last-K blocks for neo/ndbe with a **PPV@90R tail loss** (BCE warmup → +pairwise-rank +soft-pAUC@90, positive-balanced batches), **WiSE-FT** weight-interpolation for cross-center robustness, and a **multi-seed ensemble** for the deployable model. No concept stage, no comparison.

**Drive `MyDrive/RARE2026/` needs only:** `dinov2.pth`, `train.zip` (→ out/train/{images,labels}), `val.zip` (optional). The 0.zip..N.zip unlabeled chunks are NOT needed for this path.
Runtime → GPU (A6000/A100).

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip -q install timm==1.0.27 scikit-learn

In [ ]:
REPO_URL = 'https://github.com/<YOUR_USER>/viscera.git'   # <-- set
PRIVATE  = True
import os
if PRIVATE:
    from getpass import getpass; tok = getpass('GitHub token: '); REPO_URL = REPO_URL.replace('https://', f'https://{tok}@')
%cd /content
!rm -rf rare && git clone $REPO_URL rare
%cd /content/rare

In [ ]:
# extract ONLY the labeled data (train.zip/val.zip -> out/train, out/val) + dinov2.pth
from google.colab import drive; drive.mount('/content/drive')
DRIVE_DIR = '/content/drive/MyDrive/RARE2026'   # <-- your folder
import os, zipfile, shutil, glob
assert os.path.exists(f'{DRIVE_DIR}/dinov2.pth'), 'upload dinov2.pth'
shutil.copy(f'{DRIVE_DIR}/dinov2.pth', 'dinov2.pth'); print('dinov2.pth OK')
os.makedirs('out', exist_ok=True)
for name in ['train', 'val']:
    z = f'{DRIVE_DIR}/{name}.zip'
    if os.path.exists(z):
        with zipfile.ZipFile(z) as zf: zf.extractall('out')   # <name>/ at root -> out/<name>/
        print(name, '->', len(glob.glob(f'out/{name}/images/*')), 'images')
    else:
        print('MISSING', z)

In [ ]:
# labeled CSV from the JSON labels (path=out/train/images/<name>.jpg, center, label)
import json, glob, os, csv
def build_csv(split):
    rows=[]
    for j in glob.glob(f'out/{split}/labels/*.json'):
        d=json.load(open(j))
        if int(d.get('label',-1))<0: continue
        name=d.get('name', os.path.splitext(os.path.basename(j))[0])
        img=f'out/{split}/images/{name}.jpg'
        if os.path.exists(img): rows.append({'path':img,'center':d.get('center',''),'class':'','label':int(d['label'])})
    with open(f'{split}_colab.csv','w',newline='') as f:
        w=csv.DictWriter(f,fieldnames=['path','center','class','label']); w.writeheader(); w.writerows(rows)
    print(f'{split}_colab.csv', len(rows),'pos=',sum(r['label'] for r in rows),'centers=',sorted({r['center'] for r in rows}))
build_csv('train')
try: build_csv('val')
except Exception as e: print('val skip', e)

## Honest cross-center check (LOCO): fine-tune from SSL, hold out each center once

In [ ]:
for hold in ['center_2','center_1']:
    print('================ holdout', hold, '================')
    !python -m phase3.finetune --train-csv train_colab.csv --holdout {hold} \
        --unfreeze 6 --epochs 30 --bs 96 --loss bce+rank+pauc --warmup 2 --wise-ft 0.7 --out ft_{hold}.pt

## Ship: train on BOTH centers, 3 seeds → ensemble (the deployable model)

In [ ]:
for s in [0,1,2]:
    !python -m phase3.finetune --train-csv train_colab.csv --holdout none --seed {s} \
        --unfreeze 6 --epochs 30 --bs 96 --loss bce+rank+pauc --warmup 2 --wise-ft 0.7 --out ship_seed{s}.pt
import shutil
for s in [0,1,2]: shutil.copy(f'ship_seed{s}.pt', f'{DRIVE_DIR}/ship_seed{s}.pt')
print('saved 3 ensemble members to Drive')

## Inference (offline container, per-image, ensemble + hflip-TTA)
```
!python -m phase3.infer --model ship_seed0.pt,ship_seed1.pt,ship_seed2.pt --images-dir <TEST_DIR> --out preds.csv
```
Tune for more PPV@90R: sweep `--wise-ft` in {0.5,0.7,0.9} and `--unfreeze` in {4,6,8} on the LOCO check; add more seeds; the ensemble + WiSE-FT are the cross-center robustness levers.